In [169]:
from background.middlewares.service_handler import ServiceHandler
from background.modules.exchange_upbit import ExChangeUpbit
from background.modules.log import Log
from background.modules.redis import r as redis_single

from background.dao.research_result_dao import ResearchResultDao
import pandas as pd

import numpy as np
import os
import time
import datetime
from collections import defaultdict


In [173]:

class CvResearchProd03Gen01():

    def __init__(
            self,
            tickers
        ):

        self.redis = redis_single
        self.upbit = ExChangeUpbit()

        self.research_result_dao = ResearchResultDao()
        self.tickers = tickers
        self.ticker = None
        
        self.target_days_list = [
            # 31,
            # 62,
            # 93,
            #186,
            365
        ]
    
    def main(self):

        try:
            for ticker in self.tickers:
                self.ticker = ticker
                self.upbit.set_ticker(self.ticker)
                self.decide_box_zone()

        except Exception as e:
            print(f"class : {self.__class__.__name__}, error : {str(e)}")


    # ---------------------------
    # 핵심 로직
    # ---------------------------
    def decide_box_zone(self):

        
        box_zone_total = {}
        for days in self.target_days_list:
            data = self.fetch_all_ohlcv(days)
            
            box_zone_interval = {}
            for interval, df in data.items():
            
                # 1. 데이터
                closes = df['close'].values
            
                # 2. 가격 구간화 (binning)
                bins = 50  # resolution
                hist, bin_edges = np.histogram(closes, bins=bins)
            
                # 3. 밀집 구간 찾기
                density = hist / hist.sum()
            
                # 상위 밀집 구간 선택
                threshold = np.percentile(density, 50)
            
                high_density_idx = np.where(density >= threshold)[0]
            
                if len(high_density_idx) == 0:
                    return None
            
                # 4. 연속된 구간 찾기 (cluster)
                clusters = []
                current_cluster = [high_density_idx[0]]
            
                for i in range(1, len(high_density_idx)):
                    if high_density_idx[i] == high_density_idx[i-1] + 1:
                        current_cluster.append(high_density_idx[i])
                    else:
                        clusters.append(current_cluster)
                        current_cluster = [high_density_idx[i]]
            
                clusters.append(current_cluster)
                
                result = []
                tolerance = (bin_edges[1] - bin_edges[0]) * 0.2
                for cluster in clusters:
                    regi_line = bin_edges[cluster]

                    for price in regi_line:
                        matched_df = df[np.abs(df['close'] - price) <= tolerance]
                        for idx, row in matched_df.iterrows():
                            result.append({
                                "regi_line": float(price),
                                "datetime": row['datetime'].strftime('%Y-%m-%d %H:%M:%S')
                            })
                            
                box_zone_interval[interval] = result
            box_zone_total[str(days)] = box_zone_interval
            
        grouped_result = self.group_box_zone(box_zone_total)
        print(f"grouped_result : {grouped_result}")
                                    
    def fetch_all_ohlcv(self, days=30):

        count = 200
        base_intervals = [
            "day"
            #, "week"
        ]
        data = {}
        for interval in base_intervals:
                
            end_dt = datetime.datetime.now()
            start_dt = end_dt - datetime.timedelta(days=days)
            
            df = self.upbit.set_ohlcvs(interval, days, count, start_dt, end_dt)
            data[interval] = df

        return data
           
    def group_box_zone(self, box_zone_total):
        grouped_result = {}
    
        for days, intervals in box_zone_total.items():
            grouped_result[days] = {}
    
            for interval, records in intervals.items():
                temp_dict = defaultdict(list)
    
                # regi_line별로 datetime을 모으기
                for rec in records:
                    temp_dict[rec['regi_line']].append(rec['datetime'])
    
                # count 포함해서 최종 리스트 생성
                grouped_list = []
                for regi_line, datetimes in temp_dict.items():
                    grouped_list.append({
                        "regi_line": regi_line,
                        "datetimes": datetimes,
                        "count": len(datetimes)
                    })
    
                grouped_result[days][interval] = grouped_list
    
        return grouped_result

In [174]:
tickers = ["KRW-ETH"]
cvobj = CvResearchProd03Gen01(tickers)

In [175]:
cvobj.main()

grouped_result : {'365': {'day': [{'regi_line': 2210000.0, 'datetimes': ['2025-04-08 09:00:00'], 'count': 1}, {'regi_line': 2299240.0, 'datetimes': ['2025-04-07 09:00:00', '2025-04-11 09:00:00', '2025-04-16 09:00:00', '2025-04-17 09:00:00', '2025-04-18 09:00:00', '2025-04-20 09:00:00'], 'count': 6}, {'regi_line': 2566960.0, 'datetimes': ['2025-04-23 09:00:00', '2025-04-25 09:00:00', '2025-04-27 09:00:00', '2025-05-06 09:00:00', '2025-05-07 09:00:00'], 'count': 5}, {'regi_line': 2656200.0, 'datetimes': ['2025-04-02 09:00:00', '2025-04-04 09:00:00', '2025-04-05 09:00:00', '2025-05-01 09:00:00'], 'count': 4}, {'regi_line': 2745440.0, 'datetimes': ['2025-03-13 09:00:00', '2025-03-29 09:00:00', '2026-02-23 09:00:00'], 'count': 3}, {'regi_line': 2834680.0, 'datetimes': ['2025-03-12 09:00:00', '2025-03-14 09:00:00', '2025-03-15 09:00:00', '2025-03-17 09:00:00', '2025-03-18 09:00:00', '2025-03-28 09:00:00', '2025-04-01 09:00:00', '2026-02-27 09:00:00', '2026-03-01 09:00:00'], 'count': 9}, {'re